## 🔢 Finding the KEY Numeric Columns 

#### 🔃 Loading the dataset and getting only the numeric columns

In [1]:
import pandas as pd
df = pd.read_csv('./data/iris_dataset.csv')

df.info()
df.select_dtypes(include='number').columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')

#### 📃 Classifying the numeric column 

In [2]:
def classify_columns(df):
    rows=[]
    for col in df.columns:
        dtype = str(df[col].dtype)
        nuniq = df[col].nunique()
        if pd.api.types.is_numeric_dtype(df[col]):
            if col.lower().endswith(('id','code')) or nuniq == len(df):
                role = 'ID / code -> EXCLUDE from stats'
            elif nuniq <= 10:
                role = 'Few uniques -> likely coded category'
            else:
                role = 'KEY numeric (measure/ amount/ count)'
        else:
            role = 'Categorical / text / date'
        rows.append((col, dtype, nuniq, role))
    return pd.DataFrame(rows, columns=['Column', 'Dtype', 'Unique', 'Role'])

print(classify_columns(df))

              Column    Dtype  Unique                                  Role
0  sepal length (cm)  float64      35  KEY numeric (measure/ amount/ count)
1   sepal width (cm)  float64      23  KEY numeric (measure/ amount/ count)
2  petal length (cm)  float64      43  KEY numeric (measure/ amount/ count)
3   petal width (cm)  float64      22  KEY numeric (measure/ amount/ count)
4             target   object       3             Categorical / text / date


#### 🧮 Calculating descriptive statistics

In [8]:
num_cols = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
summary_stats = pd.DataFrame({
    'Mean':df[num_cols].mean(),
    'Median':df[num_cols].median(),
    'Mode': df[num_cols].mode().iloc[0],
    'Std Dev':df[num_cols].std(),
    'Min': df[num_cols].min(),
    '25th %tile (Q1)': df[num_cols].quantile(0.25),
    '50th %tile (Median)': df[num_cols].quantile(0.50),
    '75th %tile (Q3)': df[num_cols].quantile(0.75),
    '90th %tile': df[num_cols].quantile(0.90),
    'Max': df[num_cols].max()
})

print(summary_stats.round(2))

                   Mean  Median  Mode  Std Dev  Min  25th %tile (Q1)  \
sepal length (cm)  5.84    5.80   5.0     0.83  4.3              5.1   
sepal width (cm)   3.05    3.00   3.0     0.43  2.0              2.8   
petal length (cm)  3.76    4.35   1.5     1.76  1.0              1.6   
petal width (cm)   1.20    1.30   0.2     0.76  0.1              0.3   

                   50th %tile (Median)  75th %tile (Q3)  90th %tile  Max  
sepal length (cm)                 5.80              6.4        6.90  7.9  
sepal width (cm)                  3.00              3.3        3.61  4.4  
petal length (cm)                 4.35              5.1        5.80  6.9  
petal width (cm)                  1.30              1.8        2.20  2.5  


### ****Tabular Analysis from the statistics****

##### *Mean (Central Tendency & Skew Indicator)*

| Feature | Mean(cm) | Analytical Finding |
| ------- | :--------: | ------------------ |
|**Sepal Length**| 5.84 |*Highlights a specific cluster (mostly Setosa), but less useful overall because the rest of the data is widely spread.*|
|**Sepal Width**| 3.00 |*A strong, distinct peak. A massive number of flowers across different species converge exactly at 3.0 cm width, explaining the low standard deviation.*|
|**Petal Length**| 1.40 |*Crucial Finding: Exposes the multimodal nature of the data. 1.4 cm is the exact petal length of Iris-Setosa. The mode completely contradicts the mean/median, proving a massive, distinct subgroup exists at the bottom of the range.*|
|**Petal Width**| 0.20 |*Crucial Finding: Again exposes the Setosa cluster. The most common petal width is tiny, mathematically proving that aggregates alone hide the distinct species boundaries.*|

#### *Median (The "Typical Value" & Skew Anchor )* 

| Feature | Median (cm) | Analytical Finding |
| ------- |:-----------: | ------------------ |
|**Sepal Length**| 5.80 | *Represents the true middle of the dataset. If you pick a random flower, 5.8 cm is a more reliable expectation for sepal length than the mean.*|
|**Sepal Width**| 3.00 | *The exact center. Confirms that 3.0 cm is the most reliable, stable baseline for sepal width across all 150 flowers.*|
|**Petal Length**| 4.35 |*Sits comfortably in the Versicolor/Virginica range. It proves that the "middle" flower in the sorted dataset actually has quite large petals, ignoring the 50 tiny Setosa petals at the bottom.*|
|**Petal Width**| 1.30 | *Similar to petal length, the median ignores the Setosa cluster and reflects the larger species, highlighting why the mean is misleading for this column.*|

#### *Mode (The Most Frequent Value / Cluster Indicator)*

|Feature|Mode (cm) |Analytical Finding |
|-------|:----------:|-------------------|
|**Sepal Length**| 5.00 |*Highlights a specific cluster (mostly Setosa), but less useful overall because the rest of the data is widely spread.*
|**Sepal Width**| 3.00 |*A strong, distinct peak. A massive number of flowers across different species converge exactly at 3.0 cm width, explaining the low standard deviation.*|
|**Petal Length**|  1.40 |*Crucial Finding: Exposes the multimodal nature of the data. 1.4 cm is the exact petal length of Iris-Setosa. The mode completely contradicts the mean/median, proving a massive, distinct subgroup exists at the bottom of the range.*|
|**Petal Width**| 0.20 | *Crucial Finding: Again exposes the Setosa cluster. The most common petal width is tiny, mathematically proving that aggregates alone hide the distinct species boundaries.*|

#### *Standard Deviation (Spread & Relative Variability)*

| Feature|Std Dev (cm)|% of Mean|Analytical Finding|
|--------|:-------------:|:---------:|-----------------|
|**Sepal Length**|0.83|~14%|*Low relative variability. Most sepal lengths stay within a tight 1 cm band of the average.*|
|**Sepal Width**|0.43|~14%|*Lowest absolute spread. The most biologically consistent feature across all 150 flowers; almost no extreme deviations.*|
|**Petal Length**|1.76|~47%|*Highest absolute spread. Massive variability. Petal length varies wildly, making it a high-signal feature for distinguishing between species.*|
|**Petal Width**|0.76|~63%|*Highest relative variability. The width of petals scales drastically (more than half the size of the mean) depending on the species.*|

#### *Percentiles & IQR (The Middle 50% & Tails)*

|Feature|Q1 (25th %ile)|Q3 (75th %ile)|IQR (Spread)|Analytical Finding |
|-------|:--------------:|:--------------:|:------------:|-------------------|
|**Sepal Length**|5.10|6.40|1.30|*The middle 50% of flowers have sepal lengths tightly packed within a narrow 1.3 cm window.*|
|**Sepal Width**|2.80|3.30|0.50|*Tightest IQR. Half of all flowers in the entire dataset have a sepal width strictly between 2.8 and 3.3 cm. Very little diversity here.*|
|**Petal Length**|1.60|5.10|3.50|*Massive IQR. The middle 50% spans 3.5 cm. This wide gap mathematically proves there are distinct, widely separated subgroups (species) in the data, rather than one smooth bell curve.*|
|**Petal Width**|0.30|1.80|1.50|*Similar to petal length, the wide IQR confirms petal width is a high-signal feature. The jump from 0.3 to 1.8 represents the leap from Setosa to Versicolor/Virginica.*|